In [17]:
import requests
import pandas as pd
import time
import random
SIMULATION = "TNG300-1"

SNAPSHOT = 99
SUBHALO_ID = 738558

API_KEY = "6c8ec3d465c0bcf623a06a2b4c43408c"
BASE_URL = "https://www.tng-project.org/api/"

HEADERS = {
    "api-key": API_KEY
}

REQUEST_DELAY = 0.15
MAX_RETRIES = 6
cache = {}
def get_json(url):

    # Return cached result if we already requested this URL.
    if url in cache:
        return cache[url]

    for attempt in range(MAX_RETRIES):

        try:

            response = requests.get(
                url,
                headers=HEADERS,
                timeout=60
            )

            status = response.status_code

            # ------------------------------------------------
            # Successful response
            # ------------------------------------------------

            if status == 200:

                # Make sure the response is actually JSON.
                try:

                    data = response.json()

                except ValueError:

                    print()
                    print("WARNING: server returned non-JSON")
                    print("URL:", url)
                    print("Response:")
                    print(response.text[:500])

                    raise RuntimeError(
                        "TNG API returned non-JSON response."
                    )

                cache[url] = data

                time.sleep(REQUEST_DELAY)

                return data


            # ------------------------------------------------
            # Rate limiting
            # ------------------------------------------------

            elif status == 429:

                wait = 5 * (2 ** attempt)

                print(
                    f"Rate limited (429). "
                    f"Waiting {wait} seconds..."
                )

                time.sleep(wait)


            # ------------------------------------------------
            # Temporary server errors
            # ------------------------------------------------

            elif status in [500, 502, 503, 504]:

                wait = 3 * (2 ** attempt)

                print(
                    f"Temporary API error {status}. "
                    f"Waiting {wait} seconds..."
                )

                time.sleep(wait)


            # ------------------------------------------------
            # Other errors
            # ------------------------------------------------

            else:

                print()
                print("API ERROR")
                print("Status:", status)
                print("URL:", url)
                print(response.text[:1000])

                response.raise_for_status()


        except requests.exceptions.RequestException as e:

            if attempt == MAX_RETRIES - 1:
                raise

            wait = 3 * (2 ** attempt)

            print(
                f"Request failed: {e}"
            )

            print(
                f"Retrying in {wait} seconds..."
            )

            time.sleep(wait)


    raise RuntimeError(
        f"Failed to retrieve API URL after "
        f"{MAX_RETRIES} attempts:\n{url}"
    )


# ============================================================
# GET A SUBHALO
# ============================================================

def get_subhalo(snap, subhalo_id):

    url = (
        f"{BASE_URL}{SIMULATION}/snapshots/"
        f"{snap}/subhalos/{subhalo_id}/"
    )

    return get_json(url)


# ============================================================
# GET SUBLINK SIMPLE TREE
# ============================================================

def get_tree(snap, subhalo_id):

    url = (
        f"{BASE_URL}{SIMULATION}/snapshots/"
        f"{snap}/subhalos/{subhalo_id}/"
        f"sublink/simple.json"
    )

    return get_json(url)


# ============================================================
# 1. GET TARGET GALAXY'S TREE
# ============================================================

print("=" * 80)
print("GETTING TARGET SUBLINK TREE")
print("=" * 80)

target_tree = get_tree(
    SNAPSHOT,
    SUBHALO_ID
)

print(
    f"Target subhalo: {SUBHALO_ID}"
)

print(
    f"Target snapshot: {SNAPSHOT}"
)

print(
    f"Main-branch entries: "
    f"{len(target_tree['Main'])}"
)

print(
    f"Mergers: "
    f"{len(target_tree['Mergers'])}"
)


# ============================================================
# 2. MAIN PROGENITOR BRANCH
# ============================================================

# snapshot -> subhalo ID

main_branch = {
    int(snap): int(subhalo_id)
    for snap, subhalo_id
    in target_tree["Main"]
}


# ============================================================
# 3. FUNCTION TO FIND MAXIMUM STELLAR MASS
#    ALONG A SUBHALO'S MAIN PROGENITOR BRANCH
# ============================================================

def find_max_stellar_mass(
    starting_snap,
    starting_id
):

    """
    Follow the main progenitor branch of a subhalo
    and find the maximum stellar mass.

    Returns:

        max_stellar_mass
        max_mass_snapshot
        max_mass_subhalo_id
    """

    tree = get_tree(
        starting_snap,
        starting_id
    )

    branch = tree["Main"]

    max_mass = -1.0
    max_snap = None
    max_id = None

    for snap, subhalo_id in branch:

        snap = int(snap)
        subhalo_id = int(subhalo_id)

        data = get_subhalo(
            snap,
            subhalo_id
        )

        stellar_mass = data.get(
            "mass_stars"
        )

        if stellar_mass is None:
            continue

        if stellar_mass > max_mass:

            max_mass = stellar_mass
            max_snap = snap
            max_id = subhalo_id

    return (
        max_mass,
        max_snap,
        max_id
    )


# ============================================================
# 4. ANALYZE EVERY MERGER
# ============================================================

results = []

mergers = target_tree["Mergers"]

print()
print("=" * 80)
print("ANALYZING MERGERS")
print("=" * 80)


for i, merger in enumerate(mergers):

    merger_snap = int(merger[0])
    secondary_id = int(merger[1])

    print()
    print(
        f"[{i+1}/{len(mergers)}] "
        f"Merger at snapshot {merger_snap}"
    )

    print(
        f"    Secondary: {secondary_id}"
    )


    # --------------------------------------------------------
    # MAIN PROGENITOR AT THE MERGER SNAPSHOT
    # --------------------------------------------------------

    if merger_snap not in main_branch:

        print(
            "    WARNING: main branch does not contain "
            "this snapshot."
        )

        continue

    primary_id = main_branch[
        merger_snap
    ]


    # --------------------------------------------------------
    # SECONDARY'S MAIN PROGENITOR TREE
    # --------------------------------------------------------

    print(
        "    Finding secondary's maximum "
        "stellar mass..."
    )

    (
        secondary_max_mass,
        secondary_max_snap,
        secondary_max_id
    ) = find_max_stellar_mass(
        merger_snap,
        secondary_id
    )


    if secondary_max_snap is None:

        print(
            "    WARNING: could not determine "
            "secondary maximum stellar mass."
        )

        continue


    print(
        f"    Secondary max M* = "
        f"{secondary_max_mass:.6g}"
    )

    print(
        f"    At snapshot {secondary_max_snap}"
    )


    # --------------------------------------------------------
    # IMPORTANT:
    #
    # According to the TNG merger definition, evaluate
    # the PRIMARY at the SAME SNAPSHOT where the SECONDARY
    # reaches maximum stellar mass.
    #
    # --------------------------------------------------------

    if secondary_max_snap not in main_branch:

        print(
            "    WARNING: target main branch does not "
            "contain secondary max-mass snapshot."
        )

        continue


    primary_at_peak_id = main_branch[
        secondary_max_snap
    ]


    primary_peak_data = get_subhalo(
        secondary_max_snap,
        primary_at_peak_id
    )


    primary_max_time_mass = primary_peak_data.get(
        "mass_stars"
    )


    if (
        primary_max_time_mass is None
        or primary_max_time_mass <= 0
        or secondary_max_mass <= 0
    ):

        ratio = None

    else:

        ratio = (
            min(
                secondary_max_mass,
                primary_max_time_mass
            )
            /
            max(
                secondary_max_mass,
                primary_max_time_mass
            )
        )


    # --------------------------------------------------------
    # ALSO GET INSTANTANEOUS MASSES AT MERGER
    # --------------------------------------------------------

    primary_merger_data = get_subhalo(
        merger_snap,
        primary_id
    )

    secondary_merger_data = get_subhalo(
        merger_snap,
        secondary_id
    )


    primary_merger_total = (
        primary_merger_data.get("mass")
    )

    secondary_merger_total = (
        secondary_merger_data.get("mass")
    )


    if (
        primary_merger_total is not None
        and secondary_merger_total is not None
        and primary_merger_total > 0
        and secondary_merger_total > 0
    ):

        instantaneous_total_ratio = (
            min(
                primary_merger_total,
                secondary_merger_total
            )
            /
            max(
                primary_merger_total,
                secondary_merger_total
            )
        )

    else:

        instantaneous_total_ratio = None


    # --------------------------------------------------------
    # SAVE
    # --------------------------------------------------------

    results.append({

        # Target
        "target_subhalo":
            SUBHALO_ID,

        "target_snapshot":
            SNAPSHOT,

        # Merger
        "merger_snapshot":
            merger_snap,

        "primary_subhalo_at_merger":
            primary_id,

        "secondary_subhalo_at_merger":
            secondary_id,

        # Secondary maximum
        "secondary_max_stellar_mass":
            secondary_max_mass,

        "secondary_max_stellar_snapshot":
            secondary_max_snap,

        "secondary_max_stellar_subhalo":
            secondary_max_id,

        # Primary at secondary's peak
        "primary_stellar_mass_at_secondary_peak":
            primary_max_time_mass,

        "primary_subhalo_at_secondary_peak":
            primary_at_peak_id,

        # Official-style ratio
        "stellar_mass_ratio":
            ratio,

        # Instantaneous total ratio
        "instantaneous_total_mass_ratio":
            instantaneous_total_ratio,

        # Raw masses at merger
        "primary_total_mass_at_merger":
            primary_merger_total,

        "secondary_total_mass_at_merger":
            secondary_merger_total
    })


# ============================================================
# 5. DATAFRAME
# ============================================================

df = pd.DataFrame(results)

df = df.sort_values(
    "merger_snapshot"
).reset_index(drop=True)


# ============================================================
# 6. DISPLAY
# ============================================================

print()
print("=" * 110)
print("MERGER HISTORY")
print("=" * 110)

print(
    df[
        [
            "merger_snapshot",
            "primary_subhalo_at_merger",
            "secondary_subhalo_at_merger",
            "secondary_max_stellar_snapshot",
            "stellar_mass_ratio",
            "instantaneous_total_mass_ratio"
        ]
    ].to_string(index=False)
)


# ============================================================
# 7. SAVE
# ============================================================

output_file = (
    f"merger_ratios_max_stellar_"
    f"{SIMULATION}_"
    f"subhalo_{SUBHALO_ID}_"
    f"snap{SNAPSHOT}.csv"
)

df.to_csv(
    output_file,
    index=False
)

print()
print("=" * 80)
print("DONE")
print("=" * 80)

print(
    f"Successfully analyzed {len(df)} "
    f"of {len(mergers)} mergers."
)

print(
    f"Saved to: {output_file}"
)

GETTING TARGET SUBLINK TREE
Target subhalo: 738558
Target snapshot: 99
Main-branch entries: 96
Mergers: 42

ANALYZING MERGERS

[1/42] Merger at snapshot 55
    Secondary: 318069
    Finding secondary's maximum stellar mass...
    Secondary max M* = 0.043128
    At snapshot 55

[2/42] Merger at snapshot 53
    Secondary: 295538
    Finding secondary's maximum stellar mass...
    Secondary max M* = 0.386327
    At snapshot 52

[3/42] Merger at snapshot 53
    Secondary: 295552
    Finding secondary's maximum stellar mass...
    Secondary max M* = 0.185508
    At snapshot 47

[4/42] Merger at snapshot 53
    Secondary: 295721
    Finding secondary's maximum stellar mass...
    Secondary max M* = 0.001154
    At snapshot 21

[5/42] Merger at snapshot 50
    Secondary: 355723
    Finding secondary's maximum stellar mass...
    Secondary max M* = 0.009058
    At snapshot 50

[6/42] Merger at snapshot 49
    Secondary: 402338
    Finding secondary's maximum stellar mass...
    Secondary max M

RuntimeError: TNG API returned non-JSON response.

In [11]:
import requests

SIMULATION = "TNG300-1"
SNAPSHOT = 99
SUBHALO_ID = 738558
API_KEY = "6c8ec3d465c0bcf623a06a2b4c43408c"

url = (
    f"https://www.tng-project.org/api/"
    f"{SIMULATION}/snapshots/{SNAPSHOT}/subhalos/"
    f"{SUBHALO_ID}/sublink/simple.json"
)

response = requests.get(
    url,
    headers={"api-key": API_KEY}
)

print("Status code:", response.status_code)
print("Content type:", response.headers.get("Content-Type"))

if response.status_code != 200:
    print("\nAPI RESPONSE:")
    print(response.text[:2000])

else:
    data = response.json()

    print("\nKeys returned:")
    print(data.keys())

    print("\nNumber of main-branch entries:",
          len(data["Main"]))

    print("Number of mergers:",
          len(data["Mergers"]))

    print("\nMERGERS:")
    for merger in data["Mergers"]:
        print(merger)

Status code: 200
Content type: application/json

Keys returned:
dict_keys(['Main', 'Mergers'])

Number of main-branch entries: 96
Number of mergers: 42

MERGERS:
[55, 318069]
[53, 295538]
[53, 295552]
[53, 295721]
[50, 355723]
[49, 402338]
[42, 887583]
[41, 842193]
[40, 792112]
[39, 5127333]
[39, 770804]
[38, 727828]
[37, 691492]
[37, 691501]
[35, 741500]
[35, 741508]
[35, 15231339]
[35, 741513]
[34, 716514]
[32, 662874]
[33, 685544]
[33, 685550]
[33, 685543]
[33, 685547]
[31, 688918]
[32, 662872]
[31, 688925]
[29, 14405351]
[27, 776224]
[27, 8552968]
[27, 776225]
[26, 1101205]
[24, 913356]
[23, 860045]
[21, 809069]
[19, 720435]
[15, 490460]
[15, 490461]
[12, 3611376]
[11, 7579817]
[6, 2340707]
[5, 866886]


In [7]:
import io
import h5py
import numpy as np
import pandas as pd
import requests

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
API_KEY = "6c8ec3d465c0bcf623a06a2b4c43408c"  # Replace with your IllustrisTNG API key
SUBHALO_ID = 738558  # Replace with target Subhalo ID at snapshot 99
SIMULATION = "TNG300-1"
SNAPSHOT = 99


# -----------------------------------------------------------------------------
# Core Functions
# -----------------------------------------------------------------------------
def fetch_merger_tree_hdf5(
    sim: str, snap: int, subhalo_id: int, api_key: str
) -> h5py.File:
  """Downloads the SubLink merger tree HDF5 file for a given subhalo via the TNG REST API.

  Returns an in-memory h5py File object.
  """
  url = f"https://www.tng-project.org/api/{sim}/snapshots/{snap}/subhalos/{subhalo_id}/merger_tree/sublink.hdf5"
  headers = {"api-key": api_key}

  print(
      f"Fetching merger tree for {sim} Subhalo {subhalo_id} at Snap {snap}..."
  )
  response = requests.get(url, headers=headers, stream=True)
  response.raise_for_status()

  # Load HDF5 binary stream into memory
  return h5py.File(io.BytesIO(response.content), "r")


def extract_merger_ratios(
    tree_file: h5py.File,
    main_branch_only: bool = True,
    mass_type: str = "total",
) -> pd.DataFrame:
  """Traverses the SubLink tree and computes mass ratios for all merger events.

  Parameters
  ----------
  tree_file : h5py.File
      In-memory HDF5 file containing SubLink tree data.
  main_branch_only : bool, default True
      If True, traces mergers along the main progenitor line of the z=0 halo.
      If False, extracts mergers across all progenitor branches in the tree.
  mass_type : str, default 'total'
      Mass definition: 'total' (SubhaloMass) or 'stellar'
      (SubhaloMassType[:, 4]).

  Returns
  -------
  pd.DataFrame
      Table of all identified merger events and their calculated ratios.
  """
  # Extract tree dataset arrays
  snap_num = tree_file["SnapNum"][:]
  subfind_id = tree_file["SubfindID"][:]
  first_prog = tree_file["FirstProgenitorID"][:]
  next_prog = tree_file["NextProgenitorID"][:]

  # Handle mass definitions (SubLink masses are given in 10^10 M_sun / h)
  if mass_type == "stellar":
    masses = tree_file["SubhaloMassType"][:, 4]  # Index 4 corresponds to stars
  else:
    masses = tree_file["SubhaloMass"][:]  # Total mass

  merger_records = []

  def analyze_node_mergers(descendant_idx: int):
    """Checks for secondary progenitors merging into descendant_idx."""
    fp_idx = first_prog[descendant_idx]

    # fp_idx == -1 indicates a root node with no progenitors
    if fp_idx == -1:
      return

    primary_mass = masses[fp_idx]
    sec_idx = next_prog[fp_idx]

    # Iterate through all secondary progenitors joining at this snapshot
    while sec_idx != -1:
      secondary_mass = masses[sec_idx]

      # Compute mass ratio (M_secondary / M_primary)
      mass_ratio = (
          secondary_mass / primary_mass if primary_mass > 0 else np.nan
      )

      # Classify merger type based on standard astronomical criteria
      if mass_ratio >= 1.0 / 4.0:
        merger_type = "Major"
      elif mass_ratio >= 1.0 / 10.0:
        merger_type = "Minor"
      else:
        merger_type = "Micro"

      merger_records.append({
          "Descendant_Snap": snap_num[descendant_idx],
          "Descendant_SubhaloID": subfind_id[descendant_idx],
          "Primary_Snap": snap_num[fp_idx],
          "Primary_SubhaloID": subfind_id[fp_idx],
          "Primary_Mass_10_10Msun": primary_mass,
          "Secondary_Snap": snap_num[sec_idx],
          "Secondary_SubhaloID": subfind_id[sec_idx],
          "Secondary_Mass_10_10Msun": secondary_mass,
          "Mass_Ratio": mass_ratio,
          "Merger_Type": merger_type,
      })

      # Advance to the next secondary progenitor merging into the same node
      sec_idx = next_prog[sec_idx]

  if main_branch_only:
    # Walk down the main progenitor trunk from index 0 (snapshot 99 subhalo)
    curr_idx = 0
    while curr_idx != -1:
      analyze_node_mergers(curr_idx)
      curr_idx = first_prog[curr_idx]
  else:
    # Analyze all nodes across every branch in the entire tree
    for node_idx in range(len(snap_num)):
      analyze_node_mergers(node_idx)

  df = pd.DataFrame(merger_records)
  return df
import os
import tempfile
import h5py
import requests


def fetch_merger_tree_hdf5(
    sim: str, snap: int, subhalo_id: int, api_key: str
) -> h5py.File:
  """Downloads the SubLink merger tree HDF5 file to a temporary local file and opens it with h5py."""
  url = f"https://www.tng-project.org/api/{sim}/snapshots/{snap}/subhalos/{subhalo_id}/merger_tree/sublink.hdf5"
  headers = {"api-key": api_key}

  print(
      f"Fetching merger tree for {sim} Subhalo {subhalo_id} at Snap {snap}..."
  )
  response = requests.get(url, headers=headers)
  response.raise_for_status()

  content_type = response.headers.get("content-type", "")
  if "application/x-hdf5" not in content_type and "application/octet-stream" not in content_type:
    # Print the actual response body if it's JSON or HTML to diagnose
    print("Error: Received non-HDF5 response from server:")
    print(response.text[:500])
    raise ValueError(
        f"Server returned content-type '{content_type}' instead of HDF5 binary."
    )

  # Save to a temporary HDF5 file on disk (more robust on Windows)
  temp_dir = tempfile.gettempdir()
  temp_path = os.path.join(
      temp_dir, f"{sim}_{snap}_{subhalo_id}_sublink.hdf5"
  )

  with open(temp_path, "wb") as f:
    f.write(response.content)

  # Open and return the local HDF5 file object
  return h5py.File(temp_path, "r")

In [8]:
import os
import tempfile
import h5py
import requests


def fetch_merger_tree_hdf5(
    sim: str, snap: int, subhalo_id: int, api_key: str
) -> h5py.File:
  """Downloads the SubLink merger tree HDF5 file with explicit authentication and error handling."""
  url = f"https://www.tng-project.org/api/{sim}/snapshots/{snap}/subhalos/{subhalo_id}/merger_tree/sublink.hdf5"
  headers = {"api-key": api_key}

  print(
      f"Fetching merger tree for {sim} Subhalo {subhalo_id} at Snap {snap}..."
  )

  # allow_redirects=True is default, but we check if we got redirected to a login page
  response = requests.get(url, headers=headers)

  # Check if we were redirected to a login page
  if "login" in response.url or response.history:
    print(f"\n[Auth Error] Redirected to: {response.url}")
    print("Please verify that your API_KEY is correct and active.")
    raise PermissionError(
        "Authentication failed. Check your IllustrisTNG API key."
    )

  response.raise_for_status()

  content_type = response.headers.get("content-type", "")

  # If response is HTML, display the server's message
  if "text/html" in content_type:
    print("\n[Server Error Message]:")
    # Print clean text without raw HTML tags
    from bs4 import BeautifulSoup

    soup = BeautifulSoup(response.text, "html.parser")
    print(soup.get_text(strip=True)[:500])
    raise ValueError(
        f"Expected HDF5 binary, but received HTML page from {response.url}"
    )

  # Save to temporary HDF5 file
  temp_dir = tempfile.gettempdir()
  temp_path = os.path.join(temp_dir, f"{sim}_{snap}_{subhalo_id}_sublink.hdf5")

  with open(temp_path, "wb") as f:
    f.write(response.content)

  print(
      f"Successfully downloaded tree file ({len(response.content) / 1024:.1f}"
      " KB)."
  )
  return h5py.File(temp_path, "r")